In [4]:
import sys
sys.path.append("./libraries/")

import cloud_processing_lib as cpl
import glob
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from concurrent.futures import ThreadPoolExecutor
from itertools import repeat

In [5]:
def calc_day(year, month, day):
    
    #init input
    root_path = "/projekt1/ag_maahn/data_obs/lim/allsky1/images/"
    glob_query = root_path + "{0}/{1}/{2}/**/*.png".format(year,month,day) 
    path_collector = glob.glob(glob_query, recursive=True)
    path_collector.sort()

    if (dt.datetime(int(year),int(month),int(day)) - dt.datetime(2026,4,20)).total_seconds() < 0:
        obstacle_mask = cpl.prepare_mask("/home/sdoeding/analysis_scripts/auxiliary/mask_old.png")
    else:
        obstacle_mask = cpl.prepare_mask("/home/sdoeding/analysis_scripts/auxiliary/mask_new.png")
    
    mappings = cpl.calc_mappings(95, 80, 1520, "equidistant")
    duplicates = cpl.prepare_duplicates("/home/sdoeding/analysis_scripts/auxiliary/duplicates_out.csv")
    
    #init output
    timestamp_collector = []
    rb_fraction_collector = []
    brbg_fraction_collector = []
    brightness_collector = []
    
    #runtime vars
    counter=0
    start_time = dt.datetime.now()
    images_n = len(path_collector)
    
    
    #loop
    with ThreadPoolExecutor(max_workers=16) as executor:
        for res in executor.map(cpl.calc_fractions, path_collector, repeat(obstacle_mask), repeat(mappings), repeat(duplicates)):
    
            counter += 1
            
            if np.isnan(res[0]):
                continue
    
            progress_percent = np.round(counter/images_n * 100, 3)
            runtime_sec = (dt.datetime.now() - start_time).seconds
            
            print("Running {3:02d}:{4:02d} - {0} of {1} images processed ({2:.3f} %)".format(counter, images_n, progress_percent, 
                                                                                     runtime_sec // 60, runtime_sec % 60), end="\r")
    
            rb_fraction_collector.append(res[0])
            brbg_fraction_collector.append(res[1])
            timestamp_collector.append(res[2])
            brightness_collector.append(res[3])
    
    print("")
    
    stop_time = dt.datetime.now()
    runtime_sec = (stop_time - start_time).seconds
    
    print("Done")
    
    df_out = pd.DataFrame({"Timestamp":timestamp_collector, "RB_Fraction":rb_fraction_collector, "BRBG_Fraction":brbg_fraction_collector})
    df_out.to_csv("retrieval_outfiles/out_{0}{1}{2}.csv".format(year, month, day), sep=";", index="False")

In [ ]:
start_date = dt.datetime(2026, 1, 23)
end_date = dt.datetime(2026, 7, 1)
days_diff = (end_date-start_date).days


date_collector = [start_date + dt.timedelta(days=x) for x in range(0, days_diff)]

for day in date_collector:

    date_string = day.strftime("%Y%m%d")
    calc_day(date_string[0:4], date_string[4:6], date_string[6:8])

Running 00:08 - 156 of 1046 images processed (14.914 %)